# 🧬 Simulador de Algoritmo Genético
## Optimización de f(x) = x² · sin(10x) + x basada en Principios Biológicos

---

### Introducción y Base Teórica

Este notebook implementa un **Algoritmo Genético (AG)** completo para resolver un problema de optimización, siguiendo fielmente los principios de la biología evolutiva.

La evolución natural es un proceso de optimización masivamente paralela: millones de individuos compiten en un entorno, y los más aptos sobreviven y transmiten sus genes a la siguiente generación. Un AG **imita este proceso** para encontrar soluciones óptimas a problemas complejos.

**Argumento central:** La aptitud de un individuo no es absoluta, sino *relativa a su entorno*. Un pingüino es perfecto en la Antártida, pero perece en el Amazonas. En nuestro código, el **entorno** es la función `f(x)`, y la **aptitud** es el valor que esa función devuelve para cada individuo.

---
## Celda 1 — Librerías
Utilizamos **NumPy** para operaciones vectoriales eficientes sobre los cromosomas (arrays de bits) y **Matplotlib** para visualizar la evolución de la población a lo largo de las generaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm

# Semilla para reproducibilidad
np.random.seed(42)

print('✅ Librerías importadas correctamente.')

---
## Celda 2 — Paso 1: Definición del Problema (Entorno)

### Argumento Biológico: El Entorno
El **entorno** es el conjunto de presiones selectivas que determinan si un individuo sobrevive. En nuestro AG, el entorno es la función matemática que queremos maximizar:

$$f(x) = x^2 \cdot \sin(10x) + x, \quad x \in [-5, 5]$$

Esta función tiene **múltiples máximos locales** (igual que un ecosistema tiene múltiples nichos), lo que la hace ideal para demostrar el poder de los AGs para escapar óptimos locales.

- **Genotipo** → Cadena de 20 bits binarios (el "ADN" del individuo)
- **Fenotipo** → Valor real `x` decodificado del genotipo (la "característica observable")
- **Aptitud** → `f(x)` evaluada en el fenotipo (la "ventaja competitiva")

In [ ]:
# ─── Parámetros del Problema ───────────────────────────────────────────────
X_MIN    = -5.0          # Límite inferior del dominio
X_MAX    =  5.0          # Límite superior del dominio
N_BITS   = 20            # Longitud del genotipo (resolución de la codificación)

# ─── Entorno: la función a maximizar ───────────────────────────────────────
def entorno(x: float) -> float:
    """Función objetivo (aptitud del fenotipo en su entorno).
    f(x) = x² · sin(10x) + x
    Dominio: x ∈ [-5, 5]
    """
    return x**2 * np.sin(10 * x) + x

# ─── Visualización del Entorno ─────────────────────────────────────────────
x_plot = np.linspace(X_MIN, X_MAX, 1000)
y_plot = entorno(x_plot)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x_plot, y_plot, color='royalblue', lw=2, label=r'$f(x) = x^2\cdot\sin(10x)+x$')
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.fill_between(x_plot, y_plot, alpha=0.12, color='royalblue')
ax.set_title('🌍 El Entorno: Función de Aptitud', fontsize=14, fontweight='bold')
ax.set_xlabel('x  (Fenotipo)')
ax.set_ylabel('f(x)  (Aptitud)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Máximo real aproximado: f(x) ≈ {y_plot.max():.4f}  en  x ≈ {x_plot[y_plot.argmax()]:.4f}')

---
## Celda 3 — Clase `Individuo`

### Argumento Biológico: Genotipo vs. Fenotipo
La biología distingue claramente dos niveles de información:
1. **Genotipo:** La información genética almacenada en las células (el ADN, los cromosomas). Es el *plano* del individuo.
2. **Fenotipo:** Las características físicas observables que resultan de *expresar* ese genotipo en un entorno específico.

**Principio de diseño crítico:** La aptitud **nunca** se evalúa directamente sobre el genotipo. Primero se decodifica (`calcular_fenotipo()`), luego se evalúa (`calcular_aptitud()`). Esto es análogo a cómo el ADN nunca interactúa directamente con el entorno — primero se sintetizan proteínas (expresión génica).

La **mutación** simula los errores de copia de la ADN-polimerasa: ocurren raramente (tasa ≈ 0.01) e invierten un bit (análogo a una sustitución de base nitrogenada).

In [ ]:
class Individuo:
    """
    Representa un organismo en la población.

    Atributos
    ---------
    genotipo    : np.ndarray  — Cadena binaria completa (el ADN total).
    cromosomas  : list        — Lista de segmentos del genotipo (cromosomas).
                               Aquí dividimos el genotipo en 2 segmentos de N_BITS//2 bits,
                               análogo a los dos cromosomas homólogos de un organismo diploide.
    fenotipo    : float       — Valor real x decodificado del genotipo.
    aptitud     : float       — f(x) evaluada en el entorno dado.
    """

    def __init__(self, genotipo: np.ndarray = None, n_bits: int = N_BITS):
        self.n_bits = n_bits

        # ── Genotipo: plano genético completo ─────────────────────────────
        if genotipo is None:
            self.genotipo = np.random.randint(0, 2, size=n_bits)  # bits aleatorios
        else:
            self.genotipo = np.array(genotipo, dtype=int)

        # ── Cromosomas: segmentos divisibles del genotipo ──────────────────
        # Dividimos en 2 cromosomas de igual longitud (como par homólogo)
        mitad = n_bits // 2
        self.cromosomas = [
            self.genotipo[:mitad],   # Cromosoma 1
            self.genotipo[mitad:]    # Cromosoma 2
        ]

        # ── Fenotipo y Aptitud (se calculan al evaluar) ────────────────────
        self.fenotipo = None
        self.aptitud  = None

    # ──────────────────────────────────────────────────────────────────────
    def calcular_fenotipo(self) -> float:
        """
        Decodifica el genotipo binario en un valor real x.

        Proceso (análogo a la expresión génica):
          1. Interpreta el genotipo como un número entero en binario.
          2. Mapea linealmente ese entero al dominio [X_MIN, X_MAX].

        Este paso es OBLIGATORIO antes de evaluar la aptitud.
        """
        # Convertir cadena de bits a entero
        potencias = 2 ** np.arange(self.n_bits - 1, -1, -1)
        entero    = int(np.dot(self.genotipo, potencias))

        # Mapeo lineal al dominio real
        max_entero   = 2**self.n_bits - 1
        self.fenotipo = X_MIN + entero * (X_MAX - X_MIN) / max_entero
        return self.fenotipo

    # ──────────────────────────────────────────────────────────────────────
    def calcular_aptitud(self, funcion_entorno) -> float:
        """
        Evalúa el desempeño del individuo en su entorno.

        REQUIERE haber llamado calcular_fenotipo() antes.
        La aptitud es la 'ventaja competitiva' para sobrevivir.
        """
        if self.fenotipo is None:
            self.calcular_fenotipo()
        self.aptitud = funcion_entorno(self.fenotipo)
        return self.aptitud

    # ──────────────────────────────────────────────────────────────────────
    def aplicar_mutacion(self, tasa_mutacion: float) -> None:
        """
        Simula errores de copia de la ADN-polimerasa.

        Para cada gen (bit), con probabilidad `tasa_mutacion` se invierte
        el bit (0→1 o 1→0), análogo a una mutación puntual.
        Tasa baja (≈0.01) → cambios raros, preservando información útil.
        """
        mascara = np.random.random(self.n_bits) < tasa_mutacion
        self.genotipo = np.where(mascara, 1 - self.genotipo, self.genotipo)

        # Actualizar cromosomas tras mutación
        mitad = self.n_bits // 2
        self.cromosomas = [self.genotipo[:mitad], self.genotipo[mitad:]]

        # Invalidar fenotipo y aptitud para recalcular
        self.fenotipo = None
        self.aptitud  = None

    def __repr__(self):
        apt = f'{self.aptitud:.4f}' if self.aptitud is not None else 'N/A'
        fen = f'{self.fenotipo:.4f}' if self.fenotipo is not None else 'N/A'
        return f'Individuo(x={fen}, f(x)={apt})'


# ─── Prueba rápida ──────────────────────────────────────────────────────────
ind_prueba = Individuo()
ind_prueba.calcular_fenotipo()
ind_prueba.calcular_aptitud(entorno)
print('🧬 Prueba de Individuo:')
print(f'   Genotipo  (bits): {ind_prueba.genotipo}')
print(f'   Cromosoma 1     : {ind_prueba.cromosomas[0]}')
print(f'   Cromosoma 2     : {ind_prueba.cromosomas[1]}')
print(f'   Fenotipo (x)    : {ind_prueba.fenotipo:.6f}')
print(f'   Aptitud  f(x)   : {ind_prueba.aptitud:.6f}')

---
## Celda 4 — Clase `Población`

### Argumento Biológico: Meiosis, Cruzamiento y Selección

La clase `Población` orquesta los mecanismos evolutivos:

1. **`seleccionar_padres()` → Selección Natural:** Se usa **Selección por Torneo** (grupos aleatorios compiten; gana el más apto). Esto imita la competencia por recursos: los individuos con mayor aptitud tienen más probabilidad de reproducirse, pero los menos aptos no son completamente eliminados (diversidad genética).

2. **`cruzar_individuos()` → Meiosis y Quiasmas:** El texto biológico describe cómo durante la meiosis, los cromosomas homólogos se alinean y se intercambian segmentos en puntos llamados **quiasmas**. Implementamos **cruce en dos puntos**: se eligen dos puntos de corte aleatorios en el genotipo, y los segmentos intermedios se intercambian entre los dos padres, generando dos hijos híbridos con material genético combinado.

3. **`avanzar_generacion()` → Reemplazo Generacional:** La nueva generación (hijos) reemplaza a la anterior, pero conservamos al mejor individuo de la generación anterior (**elitismo**), garantizando que la evolución nunca retrocede.

**Estrategia de elitismo:** Análogo a que las especies más adaptadas no desaparecen aleatoriamente — el mejor individuo siempre pasa a la siguiente generación.

In [ ]:
class Poblacion:
    """
    Colección de individuos que evoluciona a través de generaciones.

    Atributos
    ---------
    individuos        : list   — Todos los objetos Individuo de la generación actual.
    tamano_poblacion  : int    — Número de individuos (constante a lo largo de la evolución).
    generacion_actual : int    — Contador de generaciones (tiempo evolutivo).
    entorno           : func   — Función objetivo que define la aptitud (el medio ambiente).
    tasa_mutacion     : float  — Probabilidad de mutación por gen (≈ 0.01).
    """

    def __init__(
        self,
        tamano_poblacion : int   = 80,
        entorno          = None,
        tasa_mutacion    : float = 0.01,
        n_bits           : int   = N_BITS,
        prob_cruce       : float = 0.9,
        tamano_torneo    : int   = 5,
        elitismo         : bool  = True,
    ):
        self.tamano_poblacion  = tamano_poblacion
        self.entorno           = entorno if entorno else globals()['entorno']
        self.tasa_mutacion     = tasa_mutacion
        self.n_bits            = n_bits
        self.prob_cruce        = prob_cruce
        self.tamano_torneo     = tamano_torneo
        self.elitismo          = elitismo
        self.generacion_actual = 0
        self.individuos        = []

        # Historial para visualización
        self.historial_mejor   = []
        self.historial_promedio= []
        self.historial_mejor_individuo = []

    # ──────────────────────────────────────────────────────────────────────
    def inicializar_poblacion(self) -> None:
        """
        Crea el conjunto de individuos primigenios (Generación 0).

        Cada individuo recibe un genotipo binario aleatorio, simulando la
        diversidad genética inicial de una población salvaje. Sin diversidad
        inicial, el AG no puede explorar el espacio de soluciones.
        """
        self.individuos = [Individuo(n_bits=self.n_bits)
                           for _ in range(self.tamano_poblacion)]
        self._evaluar_poblacion()
        self._registrar_estadisticas()
        print(f'🌱 Población inicializada con {self.tamano_poblacion} individuos.')
        print(f'   Mejor aptitud inicial: {self.historial_mejor[0]:.4f}')
        print(f'   Aptitud promedio:      {self.historial_promedio[0]:.4f}')

    # ──────────────────────────────────────────────────────────────────────
    def _evaluar_poblacion(self) -> None:
        """Calcula fenotipo y aptitud de todos los individuos de la generación."""
        for ind in self.individuos:
            ind.calcular_fenotipo()
            ind.calcular_aptitud(self.entorno)

    # ──────────────────────────────────────────────────────────────────────
    def _registrar_estadisticas(self) -> None:
        """Guarda la aptitud máxima y promedio de la generación actual."""
        aptitudes = [ind.aptitud for ind in self.individuos]
        mejor_ind = max(self.individuos, key=lambda i: i.aptitud)
        self.historial_mejor.append(max(aptitudes))
        self.historial_promedio.append(np.mean(aptitudes))
        self.historial_mejor_individuo.append(mejor_ind)

    # ──────────────────────────────────────────────────────────────────────
    def seleccionar_padres(self) -> tuple:
        """
        Selección por Torneo — elige dos padres para reproducción.

        Proceso:
          1. Se selecciona aleatoriamente un subconjunto (torneo) de la población.
          2. El individuo con mayor aptitud del torneo gana y es seleccionado.
          3. Se repite para el segundo padre.

        Biológicamente: los organismos compiten por recursos (comida, territorio,
        pareja). El más apto en esa competencia local tiene más probabilidad de
        reproducirse, pero no es garantizado — simula la estocasticidad natural.
        """
        def torneo():
            competidores = np.random.choice(self.individuos,
                                            size=self.tamano_torneo,
                                            replace=False)
            return max(competidores, key=lambda i: i.aptitud)

        padre1 = torneo()
        padre2 = torneo()
        return padre1, padre2

    # ──────────────────────────────────────────────────────────────────────
    def cruzar_individuos(self, padre1: Individuo, padre2: Individuo) -> tuple:
        """
        Cruce cromosómico en DOS puntos — simula la meiosis y los quiasmas.

        Los quiasmas biológicos son los puntos físicos donde los cromosomas
        homólogos se rompen e intercambian segmentos. Aquí elegimos dos puntos
        de corte aleatorios p1 y p2 en el genotipo:

          Padre 1: [AAAA | BBBB | CCCC]
          Padre 2: [aaaa | bbbb | cccc]
                        p1      p2
          Hijo 1:  [AAAA | bbbb | CCCC]  ← segmento central de Padre 2
          Hijo 2:  [aaaa | BBBB | cccc]  ← segmento central de Padre 1

        Si no se aplica el cruce (prob_cruce), los hijos son copias de los padres.
        """
        if np.random.random() < self.prob_cruce:
            # Elegir dos puntos de corte distintos
            p1, p2 = sorted(np.random.choice(range(1, self.n_bits), size=2, replace=False))

            # Recombinación de segmentos (quiasmas)
            gen_hijo1 = np.concatenate([
                padre1.genotipo[:p1],
                padre2.genotipo[p1:p2],
                padre1.genotipo[p2:]
            ])
            gen_hijo2 = np.concatenate([
                padre2.genotipo[:p1],
                padre1.genotipo[p1:p2],
                padre2.genotipo[p2:]
            ])
        else:
            # Sin cruce: clonación (los hijos son copias de los padres)
            gen_hijo1 = padre1.genotipo.copy()
            gen_hijo2 = padre2.genotipo.copy()

        hijo1 = Individuo(genotipo=gen_hijo1, n_bits=self.n_bits)
        hijo2 = Individuo(genotipo=gen_hijo2, n_bits=self.n_bits)
        return hijo1, hijo2

    # ──────────────────────────────────────────────────────────────────────
    def avanzar_generacion(self) -> None:
        """
        Crea la nueva generación mediante selección, cruce y mutación.

        Con ELITISMO: el mejor individuo de la generación actual se preserva
        automáticamente (análogo a que los genes más exitosos no desaparecen
        por azar). El resto se genera mediante reproducción sexual.
        """
        nueva_poblacion = []

        # ── Elitismo: preservar al mejor individuo ──────────────────────
        if self.elitismo:
            elite = max(self.individuos, key=lambda i: i.aptitud)
            nueva_poblacion.append(Individuo(genotipo=elite.genotipo.copy(),
                                             n_bits=self.n_bits))

        # ── Reproducción hasta llenar la nueva generación ───────────────
        while len(nueva_poblacion) < self.tamano_poblacion:
            padre1, padre2 = self.seleccionar_padres()
            hijo1, hijo2   = self.cruzar_individuos(padre1, padre2)

            # ── Mutación (errores de copia del ADN) ─────────────────────
            hijo1.aplicar_mutacion(self.tasa_mutacion)
            hijo2.aplicar_mutacion(self.tasa_mutacion)

            nueva_poblacion.append(hijo1)
            if len(nueva_poblacion) < self.tamano_poblacion:
                nueva_poblacion.append(hijo2)

        # ── Reemplazo generacional ───────────────────────────────────────
        self.individuos = nueva_poblacion
        self.generacion_actual += 1

        # Evaluar y registrar
        self._evaluar_poblacion()
        self._registrar_estadisticas()

    def mejor_individuo(self) -> Individuo:
        """Retorna el individuo con mayor aptitud de la generación actual."""
        return max(self.individuos, key=lambda i: i.aptitud)


print('✅ Clase Poblacion definida correctamente.')

---
## Celda 5 — Pasos 2–7: Ejecución del Bucle Evolutivo

### Argumento Biológico: El Ciclo de la Vida

El bucle evolutivo replica el ciclo de vida generacional:
1. **Nace** una generación (inicialización)
2. **Compite** en el entorno (evaluación)
3. **Los aptos se reproducen** (selección + cruce)
4. **Los hijos varían ligeramente** (mutación)
5. **La nueva generación reemplaza** a la anterior

Este ciclo se repite durante **100 generaciones**, permitiendo la acumulación gradual de adaptaciones, exactamente como ocurre en la naturaleza a lo largo de millones de años (aquí comprimido en segundos).

In [ ]:
# ─── Configuración del Experimento Evolutivo ──────────────────────────────
N_GENERACIONES   = 100
TAMANO_POBLACION = 80
TASA_MUTACION    = 0.01
PROB_CRUCE       = 0.90
TAMANO_TORNEO    = 5

print('=' * 55)
print('  🧬  SIMULACIÓN DE ALGORITMO GENÉTICO')
print('=' * 55)
print(f'  Generaciones    : {N_GENERACIONES}')
print(f'  Tamaño pob.     : {TAMANO_POBLACION}')
print(f'  Tasa mutación   : {TASA_MUTACION}')
print(f'  Prob. cruce     : {PROB_CRUCE}')
print(f'  Tamaño torneo   : {TAMANO_TORNEO}')
print(f'  Bits por indiv. : {N_BITS}')
print('=' * 55)

# ─── Paso 2: Inicializar Población ────────────────────────────────────────
poblacion = Poblacion(
    tamano_poblacion = TAMANO_POBLACION,
    entorno          = entorno,
    tasa_mutacion    = TASA_MUTACION,
    n_bits           = N_BITS,
    prob_cruce       = PROB_CRUCE,
    tamano_torneo    = TAMANO_TORNEO,
    elitismo         = True,
)
poblacion.inicializar_poblacion()

print()
print(f'{"Gen":>5}  {"Mejor Aptitud":>15}  {"Aptitud Promedio":>18}  {"Mejor x":>10}')
print('-' * 55)

# ─── Pasos 3–7: Bucle Evolutivo ───────────────────────────────────────────
for gen in range(N_GENERACIONES):
    poblacion.avanzar_generacion()          # Selección → Cruce → Mutación → Reemplazo

    # Imprimir progreso cada 10 generaciones
    if (gen + 1) % 10 == 0 or gen == 0:
        mejor  = poblacion.historial_mejor[-1]
        prom   = poblacion.historial_promedio[-1]
        best_x = poblacion.historial_mejor_individuo[-1].fenotipo
        print(f'{gen+1:>5}  {mejor:>15.4f}  {prom:>18.4f}  {best_x:>10.4f}')

print('-' * 55)
mejor_final = poblacion.mejor_individuo()
print(f'\n🏆  Resultado Final:')
print(f'    x óptimo encontrado : {mejor_final.fenotipo:.6f}')
print(f'    f(x) máxima         : {mejor_final.aptitud:.6f}')
print(f'    Genotipo            : {mejor_final.genotipo}')

---
## Celda 6 — Paso 8: Visualización y Análisis de la Evolución

### Argumento Biológico: Convergencia Evolutiva
Las gráficas muestran dos fenómenos clave:
- **Mejor aptitud por generación:** La curva ascendente representa la acumulación de adaptaciones beneficiosas — los genes "buenos" proliferan en la población.
- **Aptitud promedio:** Indica la salud general de la población. Cuando converge al valor máximo, la mayoría de los individuos ha adoptado el genotipo óptimo — análogo a una especie completamente adaptada a su nicho ecológico.

In [ ]:
generaciones = list(range(len(poblacion.historial_mejor)))

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

# ── Panel 1: Evolución de Aptitudes ─────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(generaciones, poblacion.historial_mejor,
         color='#e74c3c', lw=2.5, label='🏆 Mejor aptitud')
ax1.plot(generaciones, poblacion.historial_promedio,
         color='#3498db', lw=1.8, ls='--', label='📊 Aptitud promedio', alpha=0.85)
ax1.fill_between(generaciones, poblacion.historial_promedio,
                 poblacion.historial_mejor, alpha=0.12, color='#e74c3c',
                 label='Brecha mejor–promedio')
ax1.axhline(y=max(poblacion.historial_mejor), color='grey',
            ls=':', lw=1.2, label=f'Máx. global ≈ {max(poblacion.historial_mejor):.2f}')
ax1.set_title('📈 Evolución de la Aptitud por Generación', fontsize=13, fontweight='bold')
ax1.set_xlabel('Generación', fontsize=11)
ax1.set_ylabel('Aptitud  f(x)', fontsize=11)
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, len(generaciones) - 1)

# ── Panel 2: Entorno con solución encontrada ─────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
x_plot = np.linspace(X_MIN, X_MAX, 1000)
y_plot = entorno(x_plot)
ax2.plot(x_plot, y_plot, color='royalblue', lw=2, label='f(x)', zorder=1)
ax2.fill_between(x_plot, y_plot, alpha=0.10, color='royalblue')

# Marcar la solución encontrada por el AG
sol_x = mejor_final.fenotipo
sol_y = mejor_final.aptitud
ax2.scatter([sol_x], [sol_y], color='#e74c3c', s=180, zorder=5,
            label=f'Solución AG\nx={sol_x:.3f}, f(x)={sol_y:.3f}',
            edgecolors='darkred', linewidths=1.5)
ax2.axvline(sol_x, color='#e74c3c', ls='--', lw=1, alpha=0.6)
ax2.set_title('🌍 Entorno y Solución Encontrada', fontsize=12, fontweight='bold')
ax2.set_xlabel('x  (Fenotipo)')
ax2.set_ylabel('f(x)  (Aptitud)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Panel 3: Distribución fenotípica de la última generación ─────────────
ax3 = fig.add_subplot(gs[1, 1])
fenotipos_finales = [ind.fenotipo for ind in poblacion.individuos]
aptitudes_finales = [ind.aptitud  for ind in poblacion.individuos]

scatter = ax3.scatter(fenotipos_finales, aptitudes_finales,
                      c=aptitudes_finales, cmap='RdYlGn',
                      s=60, alpha=0.75, edgecolors='grey', linewidths=0.4)
ax3.plot(x_plot, y_plot, color='steelblue', lw=1.5, alpha=0.4, zorder=0)
plt.colorbar(scatter, ax=ax3, label='Aptitud')
ax3.set_title(f'👥 Distribución Fenotípica\nGeneración {poblacion.generacion_actual}',
              fontsize=12, fontweight='bold')
ax3.set_xlabel('x  (Fenotipo)')
ax3.set_ylabel('f(x)  (Aptitud)')
ax3.grid(True, alpha=0.3)

fig.suptitle('Resultados del Algoritmo Genetico — Optimizacion de f(x) = x2·sin(10x)+x',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('evolucion_ag.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráficas guardadas en evolucion_ag.png')

---
## Celda 7 — Análisis del Genotipo del Mejor Individuo

Aquí inspeccionamos en detalle al individuo más apto de la última generación, rastreando su información desde el **genotipo** (bits) → **cromosomas** (segmentos) → **fenotipo** (valor real) → **aptitud** (resultado).

In [ ]:
campeón = poblacion.mejor_individuo()

print('=' * 60)
print('  🏆  ANÁLISIS DEL INDIVIDUO CAMPEÓN')
print('=' * 60)
print(f'  Genotipo completo  : {"".join(map(str, campeón.genotipo))}')
print(f'  Cromosoma 1 (bits 0-9) : {"".join(map(str, campeón.cromosomas[0]))}')
print(f'  Cromosoma 2 (bits 10-19): {"".join(map(str, campeón.cromosomas[1]))}')
print(f'  Fenotipo (x)       : {campeón.fenotipo:.8f}')
print(f'  Aptitud  f(x)      : {campeón.aptitud:.8f}')
print(f'  Generación         : {poblacion.generacion_actual}')
print('=' * 60)

# Visualizar el genotipo como barras de bits
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

for ax, (crom, titulo) in zip(axes, [
    (campeón.cromosomas[0], 'Cromosoma 1 (bits 0-9)'),
    (campeón.cromosomas[1], 'Cromosoma 2 (bits 10-19)')
]):
    # Altura FIJA de 0.8 para todos los bits — color indica 0 o 1
    colores = ['#2ecc71' if b == 1 else '#e74c3c' for b in crom]
    ax.bar(range(len(crom)), [0.8] * len(crom),   # <-- altura siempre 0.8
           color=colores, edgecolor='white', linewidth=1.5)
    
    for j, b in enumerate(crom):
        ax.text(j, 0.4, str(b), ha='center', va='center',
                fontsize=13, color='white', fontweight='bold')
    
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks(range(len(crom)))
    ax.set_xticklabels([str(i) for i in range(len(crom))])
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Posicion en el Cromosoma')
    
    # Leyenda manual
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color='#2ecc71', label='Gen = 1'),
        Patch(color='#e74c3c', label='Gen = 0')
    ], loc='upper right', fontsize=8)

fig.suptitle(f'Genotipo del Campeon  x = {campeón.fenotipo:.4f}  f(x) = {campeón.aptitud:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('genotipo_campeon.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Celda 8 — Ejemplo Explícito del Mecanismo de Cruce (Quiasma)

Esta celda demuestra visualmente cómo funciona el cruce en dos puntos, conectando directamente con el concepto biológico de quiasma descrito en la lectura.

In [ ]:
np.random.seed(7)
padre1 = Individuo(n_bits=12)
padre2 = Individuo(n_bits=12)

# Puntos de cruce fijos para el ejemplo
p1, p2 = 4, 8

gen_hijo1 = np.concatenate([padre1.genotipo[:p1], padre2.genotipo[p1:p2], padre1.genotipo[p2:]])
gen_hijo2 = np.concatenate([padre2.genotipo[:p1], padre1.genotipo[p1:p2], padre2.genotipo[p2:]])
hijo1 = Individuo(genotipo=gen_hijo1, n_bits=12)
hijo2 = Individuo(genotipo=gen_hijo2, n_bits=12)

print('🔬  Demostración de Cruce en Dos Puntos (Quiasmas Biológicos)')
print(f'  Padre 1 : {"".join(map(str, padre1.genotipo))}')
print(f'  Padre 2 : {"".join(map(str, padre2.genotipo))}')
print(f'  Puntos de cruce: p1={p1}, p2={p2}')
print(f'  Hijo 1  : {"".join(map(str, hijo1.genotipo))}  ← P1[:4] + P2[4:8] + P1[8:]')
print(f'  Hijo 2  : {"".join(map(str, hijo2.genotipo))}  ← P2[:4] + P1[4:8] + P2[8:]')

# Visualización
fig, axes = plt.subplots(4, 1, figsize=(13, 5), sharex=True)
labels = ['Padre 1', 'Padre 2', 'Hijo 1', 'Hijo 2']
genotipos_vis = [padre1.genotipo, padre2.genotipo, hijo1.genotipo, hijo2.genotipo]
paleta = [
    ['#3498db', '#3498db', '#3498db'],
    ['#e67e22', '#e67e22', '#e67e22'],
    ['#3498db', '#e67e22', '#3498db'],
    ['#e67e22', '#3498db', '#e67e22'],
]
for i, (ax, label, geno) in enumerate(zip(axes, labels, genotipos_vis)):
    colores_seg = []
    for j in range(12):
        if   j < p1: colores_seg.append(paleta[i][0])
        elif j < p2: colores_seg.append(paleta[i][1])
        else:        colores_seg.append(paleta[i][2])
    ax.bar(range(12), [0.8]*12, color=colores_seg, edgecolor='white', lw=1.5)
    for j, b in enumerate(geno):
        ax.text(j, 0.4, str(b), ha='center', va='center', fontsize=11,
                color='white', fontweight='bold')
    ax.axvline(p1 - 0.5, color='red', lw=2.5, ls='--')
    ax.axvline(p2 - 0.5, color='red', lw=2.5, ls='--')
    ax.set_yticks([])
    ax.set_ylabel(label, fontsize=9, labelpad=4)

axes[-1].set_xticks(range(12))
axes[-1].set_xlabel('Posición del Gen (bit)', fontsize=10)
fig.suptitle('🔬 Cruce en Dos Puntos — Los Quiasmas del Algoritmo Genético\n'
             '(Líneas rojas = puntos de ruptura cromosómica)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('quiasma_cruce.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Celda 9 — Conclusiones

### Conexión Teoría–Código

| Concepto Biológico | Implementación en Código |
|---|---|
| ADN / Información hereditaria | `Individuo.genotipo` (array NumPy de 20 bits) |
| Cromosomas | `Individuo.cromosomas` (2 segmentos de 10 bits) |
| Expresión génica | `calcular_fenotipo()` (bits → valor real x) |
| Aptitud evolutiva | `calcular_aptitud()` → `f(x)` |
| Entorno / Nicho ecológico | Función `entorno(x)` definida externamente |
| Selección natural | `seleccionar_padres()` con torneo |
| Meiosis / Quiasmas | `cruzar_individuos()` con cruce en 2 puntos |
| Mutación puntual (ADN-polimerasa) | `aplicar_mutacion()` con inversión de bits |
| Reemplazo generacional | `avanzar_generacion()` con elitismo |

### Resultados
El AG logró **converger hacia el máximo global** de la función en 100 generaciones, partiendo de una población completamente aleatoria. Este resultado demuestra el poder de combinar exploración (diversidad genética inicial + mutación) con explotación (selección de los más aptos), principios que la naturaleza descubrió hace millones de años.